In [ ]:
# =====================================================================================
# GI endoscopy (Kvasir v2) - one-cell pipeline
# Kaggle download -> leakage-safe 70/15/15 split -> 2-stage fine-tuning -> rigorous test report
# =====================================================================================
import os, sys, subprocess, json, time, random
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub", "imagehash"], check=False)

# ----------------------------- CONFIG (edit here) -----------------------------------
KAGGLE_DATASET = "meetnagadia/kvasir-dataset"   # check this slug exists on Kaggle; any folder-per-class dataset works
BACKBONES      = ["efficientnetb3"]             # options: "efficientnetb3", "resnet101v2", "convnexttiny"
SEEDS          = [0]                            # training seeds; use [0, 1, 2] for mean +- std in the paper
SPLIT_SEED     = 42                             # keep fixed so every experiment uses the SAME split
NEAR_DUP_BITS  = 4                              # perceptual-hash Hamming distance (of 64) treated as near-duplicate
BATCH          = 32                            # PER GPU (global batch = 64 x number of GPUs)
STAGE1_EPOCHS  = 10                             # frozen backbone, train head only
STAGE2_EPOCHS  = 40                           # fine-tune top of backbone (early stopping decides)
UNFREEZE_FRAC  = 1.0                            # fraction of backbone layers (from the top) to fine-tune
LR_STAGE1, LR_STAGE2, WEIGHT_DECAY = 1e-3, 1e-4, 1e-4
LABEL_SMOOTH, DROPOUT, DENSE_UNITS = 0.05, 0.2, 256
CACHE_IN_RAM   = True                           # caches resized uint8 images (~2 GB); set False if RAM is tight
MIXED_PRECISION = True                          # fp16 compute: ~1.5-2x faster on T4-class GPUs
OUT_DIR        = "gi_results"
# --------------------------------------------------------------------------------------

import numpy as np, pandas as pd, tensorflow as tf
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import imagehash, kagglehub
from tensorflow import keras
from tensorflow.keras import layers

os.makedirs(OUT_DIR, exist_ok=True)
print("TF", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))
if MIXED_PRECISION:
    keras.mixed_precision.set_global_policy("mixed_float16")
strategy = tf.distribute.MirroredStrategy()     # uses all visible GPUs (falls back to 1 device)
GLOBAL_BATCH = BATCH * strategy.num_replicas_in_sync
print("Replicas:", strategy.num_replicas_in_sync, "| global batch:", GLOBAL_BATCH)
strategy2 = (tf.distribute.OneDeviceStrategy("/gpu:0") if tf.config.list_physical_devices("GPU")
             else tf.distribute.get_strategy())      # stage 2 (fine-tuning) runs on a single device

# ----------------------------- 1. DOWNLOAD ------------------------------------------
try:  # Colab secrets (KAGGLE_USERNAME / KAGGLE_KEY); otherwise env vars or ~/.kaggle/kaggle.json
    from google.colab import userdata
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        try: os.environ[k] = userdata.get(k)
        except Exception: pass
except ImportError:
    pass
data_path = kagglehub.dataset_download(KAGGLE_DATASET)
print("Downloaded to:", data_path)

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")
def find_class_root(root):
    """Return the directory whose sub-folders are the class folders (most image sub-folders wins)."""
    best, best_n = None, 0
    for d, subdirs, _ in os.walk(root):
        kids = [s for s in subdirs if any(f.lower().endswith(IMG_EXT) for f in os.listdir(os.path.join(d, s)))]
        if len(kids) > best_n:
            best, best_n = d, len(kids)
    return best
class_root = find_class_root(data_path)
class_names = sorted(s for s in os.listdir(class_root) if os.path.isdir(os.path.join(class_root, s)))
paths, labels = [], []
for ci, c in enumerate(class_names):
    for f in sorted(os.listdir(os.path.join(class_root, c))):
        if f.lower().endswith(IMG_EXT):
            paths.append(os.path.join(class_root, c, f)); labels.append(ci)
paths, labels = np.array(paths), np.array(labels)
NUM_CLASSES = len(class_names)
print(f"{len(paths)} images, {NUM_CLASSES} classes:", dict(zip(class_names, np.bincount(labels))))
if NUM_CLASSES != 8:
    print("WARNING: expected 8 Kvasir v2 classes - check KAGGLE_DATASET / folder structure.")

# ----------------------------- 2. LEAKAGE-SAFE SPLIT --------------------------------
def phash_bits(p):
    with Image.open(p) as im:
        im.draft("L", (128, 128))
        return imagehash.phash(im.convert("L")).hash.flatten()

def near_duplicate_groups(file_paths, thr):
    with ThreadPoolExecutor(8) as ex:
        bits = np.array(list(ex.map(phash_bits, file_paths)), dtype=np.float32)
    n, rows, cols = len(bits), [], []
    for i in range(0, n, 2000):                      # chunked Hamming distance via matmul
        b = bits[i:i + 2000]
        dist = bits.shape[1] - (b @ bits.T + (1 - b) @ (1 - bits).T)
        r, c = np.where(dist <= thr)
        rows.append(r + i); cols.append(c)
    rows, cols = np.concatenate(rows), np.concatenate(cols)
    g = coo_matrix((np.ones(len(rows)), (rows, cols)), shape=(n, n))
    return connected_components(g, directed=False)[1]

def group_split(y, groups, seed):
    """Stratified, group-aware 70/15/15: 20 folds -> 3 test, 3 val, 14 train. Duplicates never cross splits."""
    sgkf = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=seed)
    fold = np.zeros(len(y), int)
    for k, (_, te) in enumerate(sgkf.split(np.zeros(len(y)), y, groups)):
        fold[te] = k
    return np.where(fold < 3, "test", np.where(fold < 6, "val", "train"))

split_csv = os.path.join(OUT_DIR, "split.csv")
t0 = time.time()
groups = near_duplicate_groups(paths, NEAR_DUP_BITS)
sizes = np.bincount(groups)
print(f"Near-duplicate groups: {len(sizes)} | images in multi-member groups: {int(sizes[sizes > 1].sum())} "
      f"| largest group: {sizes.max()} ({time.time() - t0:.0f}s)")
if sizes.max() > 0.05 * len(paths):
    print("WARNING: a giant duplicate cluster formed - lower NEAR_DUP_BITS (e.g. 2).")
df = pd.DataFrame({"path": paths, "label": labels, "group": groups})
df["split"] = group_split(labels, groups, SPLIT_SEED)
df.to_csv(split_csv, index=False)
print(pd.crosstab(df["split"], [class_names[i] for i in df["label"]]))
assert not (set(df[df.split == "train"].group) & set(df[df.split == "test"].group)), "leakage!"
assert not (set(df[df.split == "val"].group) & set(df[df.split == "test"].group)), "leakage!"
assert not (set(df[df.split == "train"].group) & set(df[df.split == "val"].group)), "leakage!"

# ----------------------------- 3. MODEL ZOO -----------------------------------------
# NOTE: Keras EfficientNet/ConvNeXt normalise internally and expect raw 0-255 pixels.
#       ResNetV2 expects [-1, 1] -> done by the Rescaling layer inside the model (no train/inference mismatch).
ZOO = {
    "efficientnetb3": dict(size=300, fn=lambda **k: keras.applications.EfficientNetB3(**k), scale=(1.0, 0.0)),
    "resnet101v2":    dict(size=224, fn=lambda **k: keras.applications.ResNet101V2(**k),   scale=(1 / 127.5, -1.0)),
    "convnexttiny":   dict(size=224, fn=lambda **k: keras.applications.ConvNeXtTiny(**k),   scale=(1.0, 0.0)),
}

def make_ds(sub, size, train, seed, batch=None):
    ds = tf.data.Dataset.from_tensor_slices((sub.path.values, sub.label.values))
    def load(p, y):
        img = tf.io.decode_image(tf.io.read_file(p), channels=3, expand_animations=False)
        img.set_shape([None, None, 3])
        img = tf.cast(tf.round(tf.image.resize(img, (size, size))), tf.uint8)
        return img, tf.one_hot(y, NUM_CLASSES)
    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE)
    if CACHE_IN_RAM: ds = ds.cache()
    if train: ds = ds.shuffle(len(sub), seed=seed, reshuffle_each_iteration=True)
    return ds.batch(batch or GLOBAL_BATCH).prefetch(tf.data.AUTOTUNE)

def build_model(name):
    cfg = ZOO[name]; size = cfg["size"]
    base = cfg["fn"](include_top=False, weights="imagenet", input_shape=(size, size, 3))
    aug = keras.Sequential([                          # active only during training
        layers.RandomFlip("horizontal_and_vertical"), layers.RandomRotation(0.1),
        layers.RandomZoom(0.1), layers.RandomContrast(0.1),
        layers.RandomBrightness(0.1, value_range=(0, 255))], name="augment")
    inp = keras.Input((size, size, 3))
    x = aug(inp)
    x = layers.Rescaling(*cfg["scale"], name="rescale")(x)
    x = base(x, training=False)                       # BatchNorm stays in inference mode
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT)(x)
    x = layers.Dense(DENSE_UNITS, activation="swish", kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(DROPOUT * 0.75)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)
    return keras.Model(inp, out, name=name), base

def compile_model(model, lr, strat):
    with strat.scope():
        model.compile(optimizer=keras.optimizers.AdamW(learning_rate=lr, weight_decay=WEIGHT_DECAY),
                      loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH),
                      metrics=["accuracy"])

def callbacks(tag, patience):
    return [keras.callbacks.EarlyStopping("val_loss", patience=patience, restore_best_weights=True),
            keras.callbacks.ModelCheckpoint(os.path.join(OUT_DIR, f"{tag}.keras"), monitor="val_loss", save_best_only=True),
            keras.callbacks.ReduceLROnPlateau("val_loss", factor=0.3, patience=max(2, patience // 2), min_lr=1e-7),
            keras.callbacks.CSVLogger(os.path.join(OUT_DIR, f"{tag}_log.csv")),
            keras.callbacks.TerminateOnNaN()]

# ----------------------------- 4. EVALUATION HELPERS --------------------------------
def wilson_ci(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n)
    a = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (c - a) / d, (c + a) / d

def ece(probs, y_true, bins=15):
    conf, pred = probs.max(1), probs.argmax(1); acc = (pred == y_true).astype(float)
    edges, e = np.linspace(0, 1, bins + 1), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return e

def evaluate(model, test_df, size, tag):
    ds = make_ds(test_df, size, False, 0)
    p = model.predict(ds, verbose=0)
    p_flip = model.predict(ds.map(lambda x, y: (tf.image.flip_left_right(x), y)), verbose=0)
    p_tta = (p + p_flip) / 2
    y = test_df.label.values
    res = {}
    for name, pr in (("plain", p), ("tta", p_tta)):
        pred = pr.argmax(1); lo, hi = wilson_ci(int((pred == y).sum()), len(y))
        res[name] = dict(acc=accuracy_score(y, pred), ci95=(lo, hi), macro_f1=f1_score(y, pred, average="macro"), ece=ece(pr, y))
    pred = p_tta.argmax(1)
    print(f"\n[{tag}] TEST (n={len(y)}) plain acc {res['plain']['acc']:.4f} | TTA acc {res['tta']['acc']:.4f} "
          f"(95% CI {res['tta']['ci95'][0]:.4f}-{res['tta']['ci95'][1]:.4f}) | macro-F1 {res['tta']['macro_f1']:.4f} | ECE {res['tta']['ece']:.4f}")
    print(classification_report(y, pred, target_names=class_names, digits=4))
    plt.figure(figsize=(8, 6))
    sns.heatmap(confusion_matrix(y, pred), annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"Confusion matrix - {tag}"); plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"{tag}_confusion.png"), dpi=200); plt.show()
    return res

# ----------------------------- 5. TRAIN + EVALUATE ----------------------------------
all_results = []
train_df, val_df, test_df = (df[df.split == s].reset_index(drop=True) for s in ("train", "val", "test"))
for name in BACKBONES:
    size = ZOO[name]["size"]
    for seed in SEEDS:
        tag = f"{name}_seed{seed}"
        keras.utils.set_random_seed(seed)
        with strategy.scope():
            model, base = build_model(name)
        tr, va = make_ds(train_df, size, True, seed), make_ds(val_df, size, False, seed)

        # Stage 1: frozen backbone, train the new head
        base.trainable = False
        compile_model(model, LR_STAGE1, strategy)
        h1 = model.fit(tr, validation_data=va, epochs=STAGE1_EPOCHS, callbacks=callbacks(tag + "_s1", 4), verbose=2)

        # Stage 2: fine-tune. Changing trainable flags under MirroredStrategy breaks the gradient all-reduce
        # (shape mismatch between GPUs), so rebuild the model on ONE device and reload the stage-1 weights.
        w_path = os.path.join(OUT_DIR, f"{tag}_stage1.weights.h5")
        model.save_weights(w_path)
        del model, base; keras.backend.clear_session()
        with strategy2.scope():
            model, base = build_model(name)
            model.load_weights(w_path)
            base.trainable = True
            cut = int(len(base.layers) * (1 - UNFREEZE_FRAC))
            for i, l in enumerate(base.layers):      # top layers trainable, BatchNorm always frozen
                l.trainable = (i >= cut) and not isinstance(l, layers.BatchNormalization)
        compile_model(model, LR_STAGE2, strategy2)
        tr2, va2 = make_ds(train_df, size, True, seed, BATCH), make_ds(val_df, size, False, seed, BATCH)
        h2 = model.fit(tr2, validation_data=va2, epochs=STAGE2_EPOCHS, callbacks=callbacks(tag + "_s2", 8), verbose=2)

        # curves
        hist = {k: h1.history[k] + h2.history[k] for k in ("loss", "val_loss", "accuracy", "val_accuracy")}
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        for a, (k1, k2, t) in zip(ax, (("loss", "val_loss", "Loss"), ("accuracy", "val_accuracy", "Accuracy"))):
            a.plot(hist[k1], label="train"); a.plot(hist[k2], label="val"); a.axvline(len(h1.history["loss"]) - 1, ls="--", c="gray")
            a.set_title(f"{t} ({tag}); dashed = fine-tuning starts"); a.legend()
        plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, f"{tag}_curves.png"), dpi=200); plt.show()

        res = evaluate(model, test_df, size, tag)          # test set touched once, after training is finished
        res.update(backbone=name, seed=seed)
        all_results.append(res)
        model.save(os.path.join(OUT_DIR, f"{tag}_final.keras"))
        del model, base; keras.backend.clear_session()

# ----------------------------- 6. SUMMARY -------------------------------------------
with open(os.path.join(OUT_DIR, "results.json"), "w") as f:
    json.dump(all_results, f, indent=2, default=float)
summary = pd.DataFrame([dict(backbone=r["backbone"], seed=r["seed"], acc=r["tta"]["acc"],
                             macro_f1=r["tta"]["macro_f1"], ece=r["tta"]["ece"]) for r in all_results])
print("\n=== SUMMARY (test, with flip-TTA) ===")
print(summary.groupby("backbone")[["acc", "macro_f1", "ece"]].agg(["mean", "std"]).round(4))
print(f"\nSaved to ./{OUT_DIR}: split.csv, models (.keras), logs, curves, confusion matrices, results.json")